In [1]:
quiet_library <- function(...) { suppressPackageStartupMessages(library(...)) }

quiet_library(hise)
quiet_library(data.table)
quiet_library(dplyr)
quiet_library(purrr)

Warning message:
“package ‘data.table’ was built under R version 4.4.2”


In [2]:
if(!dir.exists("output")) {
    dir.create("output")
}

## Get sample metadata

In [3]:
sample_meta_uuid <- "af25e3e7-25c1-4476-afb4-926bd201db8f"

In [4]:
sample_meta_path <- cacheFiles(list(sample_meta_uuid))

[1] "downloading fileID af25e3e7-25c1-4476-afb4-926bd201db8f"


In [5]:
sample_meta <- read.csv(sample_meta_path, row.names = 1)

In [6]:
sample_meta <- sample_meta %>%
  mutate(subject.age_group = ifelse(
      cohort.cohortGuid == "BR1",
      "Young Adult",
      "Older Adult"
  )) %>%
  select(starts_with("cohort"), starts_with("subject"), starts_with("sample")) %>%
  select(-contains("covid"))

## Get olink reference

In [7]:
ref_uuid <- "7c2d668a-b851-4087-ba31-408de4ce889c"

In [8]:
ref_path <- cacheFiles(list(ref_uuid))

[1] "downloading fileID 7c2d668a-b851-4087-ba31-408de4ce889c"


In [9]:
ref_path

[1] "/home/workspace/input/1918706177/olink_cohort/7c2d668a-b851-4087-ba31-408de4ce889c/20201752_Skene_1820_NPX_2020-12-03_OlinkQC.csv.gz"

In [10]:
ref_olink <- fread(ref_path)

## Get bridged olink data

In [11]:
olink_uuids <- list(
    "b494cce8-1314-4f6e-9666-42f0c6e1c702", 
    "00753770-4803-4947-a290-e7469c410067", 
    "59f5f656-085f-4d0b-a240-68a9eb15e68e", 
    "1a528317-bbf3-45b7-82c9-529577cf0b15", 
    "55d5d20a-506a-459e-8315-7d66276fb8f9"
)

In [12]:
olink_paths <- cacheFiles(olink_uuids)

[1] "downloading fileID b494cce8-1314-4f6e-9666-42f0c6e1c702"
[1] "downloading fileID 00753770-4803-4947-a290-e7469c410067"
[1] "downloading fileID 59f5f656-085f-4d0b-a240-68a9eb15e68e"
[1] "downloading fileID 1a528317-bbf3-45b7-82c9-529577cf0b15"
[1] "downloading fileID 55d5d20a-506a-459e-8315-7d66276fb8f9"


In [13]:
olink_paths

[1] "/home/workspace/input/1918706177/olink_cohort/b494cce8-1314-4f6e-9666-42f0c6e1c702/AIFI-2023-06-23T22:43:24.411227655Z/q-04064/Q-04064_olink_bridged_results.csv"                                                             
[2] "/home/workspace/input/1918706177/olink_cohort/00753770-4803-4947-a290-e7469c410067/AIFI-2021-09-04T02:22:24.868949746Z/20211036-Skene_July2021Submission/bridging/20211036_Skene_NPX_2021-08-17_Bridged20210903.csv"          
[3] "/home/workspace/input/1918706177/olink_cohort/59f5f656-085f-4d0b-a240-68a9eb15e68e/AIFI-2022-02-19T01:50:28.37728978Z/20212223_olink_bridged_results.csv"                                                                     
[4] "/home/workspace/input/1918706177/olink_cohort/1a528317-bbf3-45b7-82c9-529577cf0b15/AIFI-2022-05-31T18:46:21.901486861Z/20211037-Skene_May2021Submission/bridging-results/20211037_Skene_NPX_2021-06-18_Bridged20210823_v2.csv"
[5] "/home/workspace/input/1918706177/olink_cohort/55d5d20a-506a-459e-8315-7d66276fb8f9/AIFI-2022-06-21T18:36:45.255457305Z/Q-02017_olink_bridged_results.csv"

In [14]:
olink_columns <- list(
    SampleID = 'specimen.specimenGuid',
    OlinkID = 'olink.id',
    UniProt = 'uniprot.id',
    Assay = 'olink.assay',
    MissingFreq = 'olink.missing_freq',
    Panel = 'olink.panel',
    Panel_Lot_Nr = 'olink.panel_lot_nr',
    PlateID = 'olink.plate_id',
    PlateID_qry = 'olink.plate_id',
    QC_Warning = 'olink.qc_warning',
    QC_Warning_qry = 'olink.qc_warning',
    QC_Warning_2 = 'olink.qc_warning',
    LOD = 'olink.raw_lod',
    LOD_qry = 'olink.raw_lod',
    NPX = 'olink.raw_npx',
    NPX_qry = 'olink.raw_npx',
    BatchOffset = 'olink.batch_offset',
    NPX_bridged = 'olink.norm_npx',
    LOD_bridged = 'olink.norm_lod',
    SampleKitGuid = 'sample.sampleKitGuid',
    sample.sampleKitGuid = 'sample.sampleKitGuid'
)

## Update the reference dataset

Add columns to use the same columns as bridged data

In [15]:
select_columns <- function(df, keep_columns) {
    df <- as.data.frame(df)
    df <- df[,names(df) %in% names(keep_columns)]
    names(df) <- keep_columns[names(df)]
    df <- df[,unique(unlist(keep_columns))]
    df
}

In [16]:
ref_olink <- ref_olink %>%
  as.data.frame() %>%
  mutate(BatchOffset = 0,
         NPX_bridged = NPX,
         LOD_bridged = LOD)

In [17]:
ref_olink <- select_columns(ref_olink, olink_columns)

## Select data for all bridged datasets

In [18]:
olink_list <- map(olink_paths, fread)

In [19]:
olink_list <- map(olink_list, select_columns, olink_columns)

In [20]:
all_olink <- list_rbind(c(list(ref_olink), olink_list))

In [21]:
nrow(all_olink)

[1] 3260447

In [22]:
length(unique(all_olink$sample.sampleKitGuid))

[1] 2043

In [23]:
filtered_olink <- all_olink %>%
  filter(sample.sampleKitGuid %in% sample_meta$sample.sampleKitGuid)

In [24]:
nrow(filtered_olink)

[1] 1274064

In [25]:
length(unique(filtered_olink$sample.sampleKitGuid))

[1] 867

In [26]:
missing_samples <- sample_meta %>%
  filter(!sample.sampleKitGuid %in% filtered_olink$sample.sampleKitGuid)

In [27]:
missing_samples

cohort.cohortGuid,subject.subjectGuid,subject.biologicalSex,subject.cmv,subject.bmi,subject.race,subject.ethnicity,subject.birthYear,subject.ageAtFirstDraw,subject.age_group,sample.sampleKitGuid,sample.visitName,sample.drawDate,sample.subjectAgeAtDraw,sample.daysSinceFirstVisit
<chr>,<chr>,<chr>,<chr>,<dbl>,<chr>,<chr>,<int>,<int>,<chr>,<chr>,<chr>,<chr>,<int>,<int>
BR1,BR1037,Female,Positive,33,Asian,Non-Hispanic origin,1984,36,Young Adult,KT02352,Flu Year 2 Day 7,2021-09,37,564


In [28]:
filtered_olink <- filtered_olink %>%
  left_join(sample_meta, by = 'sample.sampleKitGuid')

In [29]:
names(filtered_olink)

[1] "specimen.specimenGuid"      "olink.id"                  
 [3] "uniprot.id"                 "olink.assay"               
 [5] "olink.missing_freq"         "olink.panel"               
 [7] "olink.panel_lot_nr"         "olink.plate_id"            
 [9] "olink.qc_warning"           "olink.raw_lod"             
[11] "olink.raw_npx"              "olink.batch_offset"        
[13] "olink.norm_npx"             "olink.norm_lod"            
[15] "sample.sampleKitGuid"       "cohort.cohortGuid"         
[17] "subject.subjectGuid"        "subject.biologicalSex"     
[19] "subject.cmv"                "subject.bmi"               
[21] "subject.race"               "subject.ethnicity"         
[23] "subject.birthYear"          "subject.ageAtFirstDraw"    
[25] "subject.age_group"          "sample.visitName"          
[27] "sample.drawDate"            "sample.subjectAgeAtDraw"   
[29] "sample.daysSinceFirstVisit"

In [30]:
cohort_olink <- split(filtered_olink, filtered_olink$subject.age_group)

In [31]:
out_files <- list(
    "Young Adult" = paste0("output/sound-life_young-adult_olink_batch-corrected_", Sys.Date(), ".csv.gz"),
    "Older Adult" = paste0("output/sound-life_older-adult_olink_batch-corrected_", Sys.Date(), ".csv.gz")
)

In [32]:
walk2(
    cohort_olink, out_files[names(cohort_olink)],
    fwrite
)

## Upload data to HISE

In [33]:
study_space_uuid <- "de025812-5e73-4b3c-9c3b-6d0eac412f2a"
title <- paste("DIHA Assembled Olink for Data Apps", Sys.Date())

In [34]:
in_list <- c(list(sample_meta_uuid), list(ref_uuid), olink_uuids)

In [35]:
out_list <- out_files

In [36]:
uploadFiles(
    files = out_list,
    studySpaceId = study_space_uuid,
    title = title,
    inputFileIds = in_list,
    store = "project",
    doPrompt = FALSE
)

[1] "Cannot determine the current notebook."
[1] "1) /home/workspace/sound-life-scrna-analysis/olink-proteomics/01-R_assemble_bridged_data.ipynb"
[1] "2) /home/workspace/sound-life-scrna-analysis/01-sample_selection/00-R_Sample_Selection.ipynb"
[1] "3) /home/workspace/.local/share/Trash/files/02-R_assemble_bridged_controls.ipynb"


Please select (1-3)  1


$Message
[1] "General Okay-ness"

$VisualizationId
[1] "00000000-0000-0000-0000-000000000000"

$AbstractionId
[1] "00000000-0000-0000-0000-000000000000"

$TraceId
[1] "95e6368b-e511-4b0c-b910-c1696838a853"

$ProcessId
[1] "766eabb3-c318-48da-a29c-781b7024072f"

$WorkflowId
[1] "69d4a0c5-d3e0-43d0-9b75-81e4413ef744"

$FileIds
$FileIds[[1]]
[1] "f9656c5b-2ca6-443b-b5ed-8740187c78af"

$FileIds[[2]]
[1] "66b8017c-302b-46b8-bfbb-79b170315e0f"

In [37]:
sessionInfo()

R version 4.4.1 (2024-06-14)
Platform: x86_64-conda-linux-gnu
Running under: Ubuntu 22.04.5 LTS

Matrix products: default
BLAS/LAPACK: /home/workspace/environment/minimal/lib/libopenblasp-r0.3.28.so;  LAPACK version 3.12.0

locale:
 [1] LC_CTYPE=C.UTF-8    LC_NUMERIC=C        LC_TIME=C          
 [4] LC_COLLATE=C        LC_MONETARY=C       LC_MESSAGES=C      
 [7] LC_PAPER=C          LC_NAME=C           LC_ADDRESS=C       
[10] LC_TELEPHONE=C      LC_MEASUREMENT=C    LC_IDENTIFICATION=C

time zone: America/Los_Angeles
tzcode source: system (glibc)

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
[1] purrr_1.0.2       dplyr_1.1.4       data.table_1.16.4 hise_2.16.0      

loaded via a namespace (and not attached):
 [1] crayon_1.5.3      vctrs_0.6.5       httr_1.4.7        cli_3.6.3        
 [5] rlang_1.1.5       stringi_1.8.4     generics_0.1.3    assertthat_0.2.1 
 [9] jsonlite_1.8.9    glue_1.8.0        RCurl_